<a href="https://colab.research.google.com/github/rmndrs89/advanced-time-series-prediction/blob/main/3_Model/DeepConvLSTM_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup the notebook

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# @title Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split

In [3]:
# @title Helper functions
def compute_channel_stats(metadata, data_dir):
    channel_sums = np.zeros(3)
    channel_sq_sums = np.zeros(3)
    total_samples = 0

    for i in range(len(metadata)):
        data = np.load(f"{data_dir}/{metadata.iloc[i]['file_name']}")  # (1280, 3)
        data = data[:, 1:4].T  # (3, 1280)
        channel_sums += data.sum(axis=1)
        channel_sq_sums += (data ** 2).sum(axis=1)
        total_samples += data.shape[1]

    mean = channel_sums / total_samples
    std = np.sqrt(channel_sq_sums / total_samples - mean ** 2)
    return mean, std

In [4]:
class FOGDataset(Dataset):
    """
    FOG dataset, based on:
    https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html#dataset-class
    """
    def __init__(self, metadata, data_dir, transform=None, mean=None, std=None):
        self.metadata = metadata
        self.data_dir = data_dir
        self.transform = transform
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        file_path = f"{self.data_dir}/{self.metadata.iloc[idx]['file_name']}"
        label = int(self.metadata.iloc[idx]['label'])

        with open(file_path, 'rb') as infile:
            data = np.load(file_path)  # shape: (1280, 7)
        data = data[:, 1:4].T  # Now shape: (3, 1280) for PyTorch Conv1D

        if self.mean is not None and self.std is not None:
            data = (data - self.mean[:, None]) / self.std[:, None]

        if self.transform:
            data = self.transform(data)

        return torch.tensor(data, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [5]:
DATA_PATH = Path("/content/drive/MyDrive/Datasets/tdcsfog")
TRAIN_PATH = DATA_PATH / "train"
TEST_PATH = DATA_PATH / "test"

metadata = pd.read_csv(DATA_PATH / "train.csv")

In [6]:
# Split into train/test
train_size = int(0.8 * len(metadata))
train_meta, val_meta = train_test_split(metadata, test_size=0.2, stratify=metadata["label"], random_state=42)  # for reproducibility
mean, std = compute_channel_stats(train_meta, data_dir=TRAIN_PATH)


TypeError: unsupported format string passed to numpy.ndarray.__format__

In [8]:
with open(DATA_PATH / "train_config.npy", 'wb') as outfile:
    np.savez(outfile, mean=mean, std=std)

In [9]:
with open(DATA_PATH / "train_config.npy", 'rb') as infile:
    config = np.load(infile)
    in_mean = config['mean']
    in_std = config['std']
in_mean, in_std

(array([-9.29971751, -0.22888635,  1.95027361]),
 array([1.05952894, 1.27386502, 2.2068121 ]))

In [ ]:
train_dataset = FOGDataset(train_meta, data_dir=TRAIN_PATH)
val_dataset  = FOGDataset(val_meta, data_dir=TRAIN_PATH)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
class DeepConvLSTM(nn.Module):
    def __init__(self, input_channels=3, conv_filters=64, lstm_hidden=128, num_classes=2):
        super(DeepConvLSTM, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, conv_filters, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(conv_filters, conv_filters, kernel_size=5, padding=2),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(input_size=conv_filters, hidden_size=lstm_hidden, num_layers=2, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (batch, 3, 1280)
        x = self.conv(x)  # (batch, conv_filters, 1280)
        x = x.permute(0, 2, 1)  # (batch, 1280, conv_filters)
        lstm_out, _ = self.lstm(x)  # (batch, 1280, lstm_hidden)
        out = lstm_out[:, -1, :]  # Use last time step
        out = self.classifier(out)
        return out


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepConvLSTM().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

best_acc = 0
epochs_no_improve = 0
early_stop_patience = 7
best_model_path = 'deepconvlstm_fog.pt'

def train_model(model, loader):
    model.train()
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

def evaluate_model(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            output = model(batch_x)
            preds = output.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
    return correct / total

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
for epoch in range(5):  # up to 50 epochs
    train_model(model, train_loader)
    acc = evaluate_model(model, val_loader)
    scheduler.step(acc)

    print(f"Epoch {epoch+1}: Val Accuracy = {acc:.2%}")

    if acc > best_acc:
        best_acc = acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print("✔️  New best model saved.")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement. Patience: {epochs_no_improve}/{early_stop_patience}")

    if epochs_no_improve >= early_stop_patience:
        print("🛑 Early stopping triggered.")
        break


Epoch 1: Val Accuracy = 60.14%
✔️  New best model saved.
Epoch 2: Val Accuracy = 61.71%
✔️  New best model saved.
Epoch 3: Val Accuracy = 60.02%
⏳ No improvement. Patience: 1/7
Epoch 4: Val Accuracy = 63.89%
✔️  New best model saved.
Epoch 5: Val Accuracy = 67.87%
✔️  New best model saved.
